<a href="https://colab.research.google.com/github/Akutabox/Belajarphp/blob/main/cuaca_meteo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests, json
URL = "https://jsonplaceholder.typicode.com/posts/1"
r = requests.get(URL, timeout=10)
print("Status code :", r.status_code, r.reason)
print("Content-Type:", r.headers.get("Content-Type"))
print("Waktu respons:", r.elapsed.total_seconds(), "detik")
print("Ukuran body :", len(r.content), "byte")
print("\nBody JSON:")
print(json.dumps(r.json(), indent=2)[:400])


Status code : 200 OK
Content-Type: application/json; charset=utf-8
Waktu respons: 0.071671 detik
Ukuran body : 292 byte

Body JSON:
{
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}


In [4]:
HEADERS = {
 "Accept": "application/json",
 "User-Agent": "INSTIKI-BIFP243-Praktikum/1.0 (mahasiswa@instiki.ac.id)",
}
r = requests.get(
 "https://jsonplaceholder.typicode.com/comments",
 params={"postId": 3, "_limit": 5},
 headers=HEADERS, timeout=10,
)
print("URL final:", r.url)
print("Jumlah item:", len(r.json()))
import pandas as pd
print(pd.DataFrame(r.json())[["id", "name", "email"]])

URL final: https://jsonplaceholder.typicode.com/comments?postId=3&_limit=5
Jumlah item: 5
   id                                               name  \
0  11  fugit labore quia mollitia quas deserunt nostr...   
1  12                modi ut eos dolores illum nam dolor   
2  13  aut inventore non pariatur sit vitae voluptate...   
3  14  et officiis id praesentium hic aut ipsa dolore...   
4  15  debitis magnam hic odit aut ullam nostrum tenetur   

                            email  
0   Veronica_Goodwin@timmothy.net  
1    Oswald.Vandervort@leanne.org  
2                Kariane@jadyn.tv  
3                 Nathan@solon.io  
4  Maynard.Hodkiewicz@roberta.com  


In [6]:
def ambil(url, **kw):
    try:
        r = requests.get(url, timeout=10, **kw)
    except requests.Timeout:
        return None, "Timeout - server tidak merespons"
    except requests.ConnectionError:
        return None, "Gagal terhubung - periksa koneksi internet"

    if r.status_code == 200:
        return r.json(), "OK"
    if r.status_code == 404:
        return None, "404 - sumber daya tidak ditemukan"
    if r.status_code == 401:
        return None, "401 - autentikasi diperlukan/salah"
    if r.status_code == 429:
        return None, f"429 - rate limit, coba lagi setelah {r.headers.get('RetryAfter','?')} detik"
    if 500 <= r.status_code < 600:
        return None, f"{r.status_code} - kesalahan server, layak di-retry"
    return None, f"{r.status_code} - {r.reason}"

for u in ["https://jsonplaceholder.typicode.com/posts/1",
          "https://jsonplaceholder.typicode.com/tidakada",
          "https://httpstat.us/500"]:
    data, pesan = ambil(u)
    print(f"{u[:55]:57s} -> {pesan}")

https://jsonplaceholder.typicode.com/posts/1              -> OK
https://jsonplaceholder.typicode.com/tidakada             -> 404 - sumber daya tidak ditemukan
https://httpstat.us/500                                   -> Gagal terhubung - periksa koneksi internet


In [21]:
import time
import requests

def ambil_dengan_retry(url, maks_percobaan=4, jeda_awal=1.0, **kw):
    jeda = jeda_awal
    for percobaan in range(1, maks_percobaan + 1):
        try:
            r = requests.get(url, timeout=10, **kw)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:
                jeda_server = float(r.headers.get("Retry-After", jeda))
                print(f" [percobaan {percobaan}] 429, tunggu {jeda_server}s")
                time.sleep(jeda_server)
            elif 500 <= r.status_code < 600:
                print(f" [percobaan {percobaan}] {r.status_code}, tunggu {jeda}s")
                time.sleep(jeda)
            else:
                raise RuntimeError(f"Tidak dapat dipulihkan: {r.status_code}")
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f" [percobaan {percobaan}] {type(e).__name__}, tunggu {jeda}s")
            time.sleep(jeda)
        jeda *= 2 # exponential backoff
    raise RuntimeError(f"Gagal setelah {maks_percobaan} percobaan: {url}")

data = ambil_dengan_retry("https://jsonplaceholder.typicode.com/users")
print("Berhasil, jumlah user:", len(data))

Berhasil, jumlah user: 10


In [20]:
import requests
import time

def ambil_semua_halaman(base_url, limit=20, maks_halaman=5, jeda=0.5):
    semua, halaman = [], 1
    while halaman <= maks_halaman:
        r = requests.get(base_url, params={"_page": halaman, "_limit": limit},
                         headers=HEADERS, timeout=10)
        batch = r.json()
        if not batch:
            break
        semua.extend(batch)
        print(f"halaman {halaman}: +{len(batch)} item (total {len(semua)})")
        if len(batch) < limit:
            break

        halaman += 1
        time.sleep(jeda) # bersikap sopan pada server
    return semua

posts = ambil_semua_halaman("https://jsonplaceholder.typicode.com/posts")
print("\nTotal terkumpul:", len(posts))

halaman 1: +20 item (total 20)
halaman 2: +20 item (total 40)
halaman 3: +20 item (total 60)
halaman 4: +20 item (total 80)
halaman 5: +20 item (total 100)

Total terkumpul: 100


In [8]:
from pandas import json_normalize
from pathlib import Path
from datetime import datetime
import json
users = requests.get("https://jsonplaceholder.typicode.com/users",
 headers=HEADERS, timeout=10).json()
# Simpan raw
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path(f"data/raw/api_users_{stamp}.json").write_text(
 json.dumps(users, indent=2), encoding="utf-8")
# Flatten struktur bersarang address & company
flat = json_normalize(users, sep="_")
print("Kolom hasil flatten:")
print(list(flat.columns))
kolom = ["id", "name", "username", "email",
 "address_city", "address_geo_lat", "address_geo_lng", "company_name"]
tabel = flat[kolom]
Path("data/interim").mkdir(parents=True, exist_ok=True)
tabel.to_csv("data/interim/api_users.csv", index=False)
print(tabel.head())

Kolom hasil flatten:
['id', 'name', 'username', 'email', 'phone', 'website', 'address_street', 'address_suite', 'address_city', 'address_zipcode', 'address_geo_lat', 'address_geo_lng', 'company_name', 'company_catchPhrase', 'company_bs']
   id              name   username                      email   address_city  \
0   1     Leanne Graham       Bret          Sincere@april.biz    Gwenborough   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv    Wisokyburgh   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net  McKenziehaven   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org    South Elvis   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca     Roscoeview   

  address_geo_lat address_geo_lng        company_name  
0        -37.3159         81.1496     Romaguera-Crona  
1        -43.9509        -34.4618        Deckow-Crist  
2        -68.6102        -47.0653  Romaguera-Jacobson  
3         29.4572       -164.2990       Robel-Corkery  
4

In [9]:
import os
# Cara 1: environment variable (disarankan)
API_KEY = os.getenv("API_KEY_PRAKTIKUM")
print("API key terbaca dari environment:", bool(API_KEY))
# Cara 2: berkas .env yang MASUK ke .gitignore
# isi .env : API_KEY_PRAKTIKUM=abc123
# from dotenv import load_dotenv; load_dotenv()
# Tambahkan ke .gitignore
from pathlib import Path
gi = Path(".gitignore")
isi = gi.read_text() if gi.exists() else ""
for pola in [".env", "*.key", "credentials.json"]:
 if pola not in isi:
   isi += pola + "\n"
gi.write_text(isi)
print("\n.gitignore:\n" + isi)

API key terbaca dari environment: False

.gitignore:
.env
*.key
credentials.json



In [10]:
{
    "latitude": 52.52,
    "longitude": 13.419,
    "elevation": 44.812,
    "generationtime_ms": 2.2119,
    "utc_offset_seconds": 0,
    "timezone": "Europe/Berlin",
    "timezone_abbreviation": "CEST",
    "hourly": {
        "time": ["2022-07-01T00:00", "2022-07-01T01:00", "2022-07-01T02:00", ...],
        "temperature_2m": [13, 12.7, 12.7, 12.5, 12.5, 12.8, 13, 12.9, 13.3, ...]
    },
    "hourly_units": {
        "temperature_2m": "°C"
    }
}

{'latitude': 52.52,
 'longitude': 13.419,
 'elevation': 44.812,
 'generationtime_ms': 2.2119,
 'utc_offset_seconds': 0,
 'timezone': 'Europe/Berlin',
 'timezone_abbreviation': 'CEST',
 'hourly': {'time': ['2022-07-01T00:00',
   '2022-07-01T01:00',
   '2022-07-01T02:00',
   Ellipsis],
  'temperature_2m': [13,
   12.7,
   12.7,
   12.5,
   12.5,
   12.8,
   13,
   12.9,
   13.3,
   Ellipsis]},
 'hourly_units': {'temperature_2m': '°C'}}

In [12]:
!pip install openmeteo-requests retry-requests requests-cache

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -8.65,
	"longitude": 115.2167,
	"daily": "weather_code",
	"hourly": "temperature_2m",
	"current": "temperature_2m",
	"past_days": 7,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {abs(response.Latitude())}°{'S' if response.Latitude() < 0 else 'N'} {abs(response.Longitude())}°{'W' if response.Longitude() < 0 else 'E'}")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_temperature_2m = current.Variables(0).Value()

print(f"\nCurrent time: {current.Time()}")
print(f"Current temperature_2m: {current_temperature_2m}")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	)
}

daily_data["weather_code"] = daily_weather_code

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.7/890.7 kB 23.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.1 MB/s  0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [openmeteo-requests]
Coordinates: 8.68189811706543°S 115.19379425048828°E
Elevation: 35.0 m asl
Timezone difference to GMT+0: 0s

Current time: 1794271500
Current temperature_2m: 29.049999237060547

Hourly data
                          date  temperature_2m
0   2026-11-03 00:00:00+00:00       27.150000
1   2026-11-03 01:00:00+00:00       28.299999
2   2026-11-03 02:00:00+00:00       29.049999
3   2026-11-03 03:00:00+00:00       29.650000
4   2026-11-03 04:00:00+00:00       30.100000
..                        ...             ...
331 2026-11-16 19:00:00+00:00       25.150000
332 2026-11-16 20:00:00+00:00       2

In [16]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -8.65,
	"longitude": 115.2167,
	"daily": "weather_code",
	"hourly": "temperature_2m",
	"current": "temperature_2m",
	"past_days": 7,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {abs(response.Latitude())}°{'S' if response.Latitude() < 0 else 'N'} {abs(response.Longitude())}°{'W' if response.Longitude() < 0 else 'E'}")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_temperature_2m = current.Variables(0).Value()

print(f"\nCurrent time: {current.Time()}")
print(f"Current temperature_2m: {current_temperature_2m}")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	)
}

daily_data["weather_code"] = daily_weather_code

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

Coordinates: 8.68189811706543°S 115.19379425048828°E
Elevation: 35.0 m asl
Timezone difference to GMT+0: 0s

Current time: 1794271500
Current temperature_2m: 29.049999237060547

Hourly data
                          date  temperature_2m
0   2026-11-03 00:00:00+00:00       27.150000
1   2026-11-03 01:00:00+00:00       28.299999
2   2026-11-03 02:00:00+00:00       29.049999
3   2026-11-03 03:00:00+00:00       29.650000
4   2026-11-03 04:00:00+00:00       30.100000
..                        ...             ...
331 2026-11-16 19:00:00+00:00       25.150000
332 2026-11-16 20:00:00+00:00       25.500000
333 2026-11-16 21:00:00+00:00       26.000000
334 2026-11-16 22:00:00+00:00       26.500000
335 2026-11-16 23:00:00+00:00       27.049999

[336 rows x 2 columns]

Daily data
                         date  weather_code
0  2026-11-03 00:00:00+00:00          53.0
1  2026-11-04 00:00:00+00:00          51.0
2  2026-11-05 00:00:00+00:00          55.0
3  2026-11-06 00:00:00+00:00          51.0
4  20

In [18]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -8.65,
	"longitude": 115.2167,
	"daily": ["weather_code", "temperature_2m_min", "temperature_2m_max", "rain_sum", "sunrise", "sunset", "daylight_duration", "sunshine_duration", "moonrise", "moonset", "moon_phase"],
	"hourly": ["temperature_2m", "rain", "weather_code", "temperature_80m", "temperature_120m", "temperature_180m"],
	"current": ["temperature_2m", "weather_code", "rain", "cloud_cover", "pressure_msl", "surface_pressure", "apparent_temperature"],
	"timezone": "Asia/Singapore",
	"past_days": 7,
}
responses = openmeteo.weather_api(url, params = params)

# Days without a moonrise or moonset return this value instead of a timestamp
missing_int64 = 9223372036854775807

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {abs(response.Latitude())}°{'S' if response.Latitude() < 0 else 'N'} {abs(response.Longitude())}°{'W' if response.Longitude() < 0 else 'E'}")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_temperature_2m = current.Variables(0).Value()
current_weather_code = current.Variables(1).Value()
current_rain = current.Variables(2).Value()
current_cloud_cover = current.Variables(3).Value()
current_pressure_msl = current.Variables(4).Value()
current_surface_pressure = current.Variables(5).Value()
current_apparent_temperature = current.Variables(6).Value()

print(f"\nCurrent time: {current.Time()}")
print(f"Current temperature_2m: {current_temperature_2m}")
print(f"Current weather_code: {current_weather_code}")
print(f"Current rain: {current_rain}")
print(f"Current cloud_cover: {current_cloud_cover}")
print(f"Current pressure_msl: {current_pressure_msl}")
print(f"Current surface_pressure: {current_surface_pressure}")
print(f"Current apparent_temperature: {current_apparent_temperature}")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_rain = hourly.Variables(1).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(2).ValuesAsNumpy()
hourly_temperature_80m = hourly.Variables(3).ValuesAsNumpy()
hourly_temperature_120m = hourly.Variables(4).ValuesAsNumpy()
hourly_temperature_180m = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["rain"] = hourly_rain
hourly_data["weather_code"] = hourly_weather_code
hourly_data["temperature_80m"] = hourly_temperature_80m
hourly_data["temperature_120m"] = hourly_temperature_120m
hourly_data["temperature_180m"] = hourly_temperature_180m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(2).ValuesAsNumpy()
daily_rain_sum = daily.Variables(3).ValuesAsNumpy()
daily_sunrise = daily.Variables(4).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(5).ValuesInt64AsNumpy()
daily_daylight_duration = daily.Variables(6).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(7).ValuesAsNumpy()
daily_moonrise = daily.Variables(8).ValuesInt64AsNumpy()
daily_moonset = daily.Variables(9).ValuesInt64AsNumpy()
daily_moon_phase = daily.Variables(10).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["weather_code"] = daily_weather_code
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["rain_sum"] = daily_rain_sum
daily_data["sunrise"] = pd.to_datetime(daily_sunrise, unit = "s", utc = True).tz_convert(response.Timezone().decode())
daily_data["sunset"] = pd.to_datetime(daily_sunset, unit = "s", utc = True).tz_convert(response.Timezone().decode())
daily_data["daylight_duration"] = daily_daylight_duration
daily_data["sunshine_duration"] = daily_sunshine_duration
daily_moonrise = pd.Series(daily_moonrise).mask(daily_moonrise == missing_int64)
daily_data["moonrise"] = pd.to_datetime(daily_moonrise, unit = "s", utc = True).dt.tz_convert(response.Timezone().decode())
daily_moonset = pd.Series(daily_moonset).mask(daily_moonset == missing_int64)
daily_data["moonset"] = pd.to_datetime(daily_moonset, unit = "s", utc = True).dt.tz_convert(response.Timezone().decode())
daily_data["moon_phase"] = daily_moon_phase

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

Coordinates: 8.68189811706543°S 115.19379425048828°E
Elevation: 35.0 m asl
Timezone: b'Asia/Singapore'b'GMT+8'
Timezone difference to GMT+0: 28800s

Current time: 1794273300
Current temperature_2m: 29.75
Current weather_code: 3.0
Current rain: 0.0
Current cloud_cover: 96.0
Current pressure_msl: 1012.0
Current surface_pressure: 1008.0145874023438
Current apparent_temperature: 36.032020568847656

Hourly data
                          date  temperature_2m  rain  weather_code  \
0   2026-11-03 00:00:00+08:00       24.900000   0.0           1.0   
1   2026-11-03 01:00:00+08:00       24.400000   0.0           1.0   
2   2026-11-03 02:00:00+08:00       24.350000   0.0          45.0   
3   2026-11-03 03:00:00+08:00       24.250000   0.0          45.0   
4   2026-11-03 04:00:00+08:00       24.049999   0.0          45.0   
..                        ...             ...   ...           ...   
331 2026-11-16 19:00:00+08:00       27.700001   0.1          51.0   
332 2026-11-16 20:00:00+08:00       2